In [4]:
import numpy as np
import pandas as pd
import joblib

# Load model & preprocessor
model = joblib.load("random_forest_heat_exchanger_model.pkl")
preprocessor = joblib.load("preprocessor_dual_fluid.pkl")

In [ ]:
def optimize_parameters(
    tube_fluid,
    shell_fluid,
    Q_required,
    n_trials=5000
):
    candidates = []

    for _ in range(n_trials):

        candidate = {
            "tube_fluid": tube_fluid,
            "shell_fluid": shell_fluid,

            "shell_diameter": np.random.uniform(0.52, 0.58),
            "tube_outer_diameter": np.random.uniform(0.023, 0.027),
            "tube_inner_diameter": np.random.uniform(0.019, 0.023),
            "tube_length": np.random.uniform(2.5, 3.5),
            "number_of_tubes": np.random.randint(18, 26),
            "tube_pitch": np.random.uniform(0.03, 0.035),

            "baffle_spacing": np.random.uniform(0.15, 0.25),
            "number_of_baffles": np.random.randint(8, 12),

            "tube_mass_flow_rate": np.random.uniform(0.5, 4.0),
            "shell_mass_flow_rate": np.random.uniform(0.5, 3.0),

            "tube_inlet_temp": 420,
            "tube_outlet_temp": np.random.uniform(360, 400),

            "shell_inlet_temp": 330,
            "shell_outlet_temp": np.random.uniform(350, 380),

            "tube_thermal_conductivity": np.random.choice([16, 205, 385])
        }

        X = pd.DataFrame([candidate])
        Xp = preprocessor.transform(X)
        Q_pred = np.expm1(model.predict(Xp))[0]

        if Q_pred >= Q_required:
            candidate["Q_pred"] = Q_pred
            candidates.append(candidate)

    if not candidates:
        return None

    df = pd.DataFrame(candidates)
    return df.sort_values("Q_pred", ascending=False).head(3)


In [3]:
best_designs = optimize_parameters(
    tube_fluid="water",
    shell_fluid="oil",
    Q_required=12000  # 12 kW
)

print(best_designs)


     tube_fluid shell_fluid  shell_diameter  tube_outer_diameter  \
2064      water         oil        0.579717             0.025937   
509       water         oil        0.556784             0.026786   
2165      water         oil        0.579760             0.026870   

      tube_inner_diameter  tube_length  number_of_tubes  tube_pitch  \
2064             0.020133     3.370929               24    0.033536   
509              0.022672     3.258500               25    0.031874   
2165             0.022962     3.498744               24    0.032550   

      baffle_spacing  number_of_baffles  tube_mass_flow_rate  \
2064        0.164569                  8             3.599792   
509         0.162837                 10             2.680058   
2165        0.205653                 11             3.708040   

      shell_mass_flow_rate  tube_inlet_temp  tube_outlet_temp  \
2064              2.763186              420        382.683963   
509               2.884484              420        397.